# Task 1: News Topic Classifier Using BERT

## Objective

The objective of this project is to build a News Topic Classification model using the BERT transformer model. The model is trained on the AG News dataset to classify news headlines into one of four categories:

- World
- Sports
- Business
- Science/Technology

The project demonstrates Natural Language Processing (NLP), transfer learning, transformer fine-tuning, and model deployment using Gradio.

## Step 1: Import Required Libraries

Import all necessary libraries that will be used throughout the project.

These libraries provide tools for loading datasets, preprocessing text, loading the BERT model, training, evaluation, and numerical computations.

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer
import evaluate
import numpy as np

## Step 2: Load the AG News Dataset

Load the AG News dataset directly from Hugging Face.

The dataset contains:

- Training data
- Testing data

Each record contains:

- News headline
- Category label

In [2]:
dataset = load_dataset("ag_news")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


## Step 3: Explore the Dataset

Display one sample from the dataset to understand its structure.

Each sample consists of:

- News headline
- Corresponding category label

This helps us understand the data before training the model.

In [3]:
print(dataset["train"][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## Step 4: Load the BERT Tokenizer

BERT cannot process plain English text directly.

The tokenizer converts text into numerical tokens that can be understood by the BERT model.

This step prepares the text for machine learning.

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

## Step 5: Define the Tokenization Function

Create a function that converts every news headline into tokens.

Two preprocessing techniques are applied:

- **Truncation:** Shortens long sentences.
- **Padding:** Makes all input sequences the same length.

In [5]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length"
    )

## Step 6: Tokenize the Dataset

Apply the tokenizer to every news headline in the dataset.

The original text is converted into numerical tokens that will be used as input for the BERT model.

In [6]:
tokenized_dataset = dataset.map(tokenize, batched=True)

## Step 7: Load the Pre-trained BERT Model

Load the BERT Base model that has already been trained on a large amount of English text.

A classification layer is added to predict one of the four news categories.

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step 8: Load Evaluation Metrics

Load the evaluation metrics that will measure the model's performance.

The following metrics are used:

- Accuracy
- F1 Score

These metrics help determine how well the classifier performs.

In [8]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

## Step 9: Define the Evaluation Function

Create a custom evaluation function.

This function compares the model's predictions with the true labels and calculates:

- Accuracy
- Weighted F1 Score

These values are displayed after every evaluation.

In [9]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

## Step 10: Configure Training Parameters

Define the training settings for the BERT model.

Important parameters include:

- Learning rate
- Batch size
- Number of training epochs
- Model saving strategy
- Evaluation strategy

These settings control how the model learns from the data.

In [10]:
training_args = TrainingArguments(
    output_dir="./bert_news",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01
)

E:\anaconda\Lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Step 11: Create the Trainer

The Trainer object combines:

- The BERT model
- Training dataset
- Testing dataset
- Training parameters
- Evaluation function

It manages the complete training and evaluation process automatically.

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].shuffle(seed=42).select(range(1000)),
    eval_dataset=tokenized_dataset["test"].shuffle(seed=42).select(range(200)),
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

C:\Users\wwwha\AppData\Local\Temp\ipykernel_18548\3630123623.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## Step 12: Train the BERT Model

Train the BERT model using the AG News training dataset.

During training, the model learns patterns in the news headlines and improves its predictions after every iteration.

In [12]:
trainer.train()

E:\anaconda\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.598254,0.815000,0.814596


E:\anaconda\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=125, training_loss=0.8676544799804687, metrics={'train_runtime': 6266.5817, 'train_samples_per_second': 0.16, 'train_steps_per_second': 0.02, 'total_flos': 263115780096000.0, 'train_loss': 0.8676544799804687, 'epoch': 1.0})

##### Step 13: Evaluate the Model

Evaluate the trained model using the testing dataset.

The model reports important performance metrics such as:

- Accuracy
- F1 Score

Higher values indicate better classification performance.

In [13]:
trainer.evaluate()

E:\anaconda\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.5982542037963867,
 'eval_accuracy': 0.815,
 'eval_f1': 0.8145955013210957,
 'eval_runtime': 305.5368,
 'eval_samples_per_second': 0.655,
 'eval_steps_per_second': 0.082,
 'epoch': 1.0}

## Step 14: Save the Trained Model

Save the trained BERT model to the local system.

The saved model can later be loaded without retraining.

In [14]:
trainer.save_model("bert_news_classifier")

## Step 15: Predict the Category of New News Headlines

Test the trained model using a new news headline.

The model predicts the most likely category among the four available classes.

In [15]:
text = "NASA discovers a new planet."

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

prediction = outputs.logits.argmax().item()

labels = [
    "World",
    "Sports",
    "Business",
    "Science/Technology"
]

print(labels[prediction])

Science/Technology


## Step 16: Deploy the Model Using Gradio

Create a simple web-based interface that allows users to enter a news headline and receive the predicted category instantly.

This makes the trained model interactive and easy to use without writing additional Python code.

In [16]:
import gradio as gr

labels = [
    "World",
    "Sports",
    "Business",
    "Science/Technology"
]

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    prediction = outputs.logits.argmax().item()
    return labels[prediction]

demo = gr.Interface(
    fn=predict,
    inputs="text",
    outputs="text",
    title="News Topic Classifier Using BERT"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# Conclusion

In this project, a BERT-based News Topic Classifier was developed using the AG News dataset.

The workflow included:

- Loading and exploring the dataset
- Text preprocessing using the BERT tokenizer
- Fine-tuning the pre-trained BERT model
- Evaluating the model using Accuracy and F1 Score
- Predicting categories for unseen news headlines
- Deploying the model with Gradio

This project demonstrates the practical application of transformer-based Natural Language Processing (NLP) for text classification.